In [9]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ============================================================
# HYBRID ISOLATION FOREST + AUTOENCODER
# ============================================================

import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
import tensorflow as tf

from tensorflow import keras

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/intentmap-nids/intentmap-nids"
)

DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
REPORT_DIR = BASE_DIR / "reports"
CONFIG_DIR = BASE_DIR / "config"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("CONFIG_DIR:", CONFIG_DIR)

BASE_DIR: /content/drive/MyDrive/intentmap-nids/intentmap-nids
DATA_DIR: /content/drive/MyDrive/intentmap-nids/intentmap-nids/data/processed
MODEL_DIR: /content/drive/MyDrive/intentmap-nids/intentmap-nids/models
REPORT_DIR: /content/drive/MyDrive/intentmap-nids/intentmap-nids/reports
CONFIG_DIR: /content/drive/MyDrive/intentmap-nids/intentmap-nids/config


In [11]:
# ============================================================
# REQUIRED FILES
# ============================================================

VAL_FILE = DATA_DIR / "val_normal.npz"
TEST_FILE = DATA_DIR / "test_full.npz"

IF_MODEL_FILE = (
    MODEL_DIR / "isolation_forest_candidate1.joblib"
)

AE_MODEL_FILE = (
    MODEL_DIR / "autoencoder_candidate1.keras"
)

IF_CONFIG_FILE = (
    CONFIG_DIR / "isolation_forest_candidate1.json"
)

AE_CONFIG_FILE = (
    CONFIG_DIR / "autoencoder_candidate1.json"
)

required_files = [
    VAL_FILE,
    TEST_FILE,
    IF_MODEL_FILE,
    AE_MODEL_FILE
]

print("HYBRID INPUT CHECK")
print("==================")

all_found = True

for file_path in required_files:

    exists = file_path.exists()

    print(
        file_path.name,
        "->",
        exists
    )

    if not exists:
        all_found = False


if all_found:
    print("\nAll required files are ready.")
else:
    print("\nERROR: One or more required files are missing.")

HYBRID INPUT CHECK
val_normal.npz -> True
test_full.npz -> True
isolation_forest_candidate1.joblib -> False
autoencoder_candidate1.keras -> True

ERROR: One or more required files are missing.


In [12]:
# ============================================================
# LOAD PROCESSED DATA
# ============================================================

def load_npz(path):

    data = np.load(path)

    X = data["X"]

    y = (
        data["y"]
        if "y" in data.files
        else None
    )

    return X, y


X_val, _ = load_npz(
    VAL_FILE
)

X_test, y_test = load_npz(
    TEST_FILE
)

# TensorFlow-friendly format
X_val = X_val.astype("float32")
X_test = X_test.astype("float32")

y_test = np.asarray(
    y_test
).reshape(-1).astype(int)

print(
    "Normal validation:",
    X_val.shape
)

print(
    "Full test:",
    X_test.shape
)

print(
    "Normal test records:",
    np.sum(y_test == 0)
)

print(
    "Attack test records:",
    np.sum(y_test == 1)
)

Normal validation: (10266, 41)
Full test: (82332, 41)
Normal test records: 37000
Attack test records: 45332


In [15]:
# ============================================================
# LOAD PRE-TRAINED MODELS
# ============================================================

import os
import joblib
from tensorflow import keras

# Correct Isolation Forest path
IF_MODEL_FILE = (
    "/content/drive/MyDrive/intentmap-nids/intentmap-nids/"
    "data/processed/models/isolation_forest_candidate1.joblib"
)

# Check files before loading
print("IF model exists:", os.path.exists(IF_MODEL_FILE))
print("AE model exists:", os.path.exists(AE_MODEL_FILE))

# ------------------------------------------------------------
# Load Isolation Forest
# ------------------------------------------------------------
print("\nLoading Isolation Forest...")

if_model = joblib.load(IF_MODEL_FILE)

print("Isolation Forest loaded successfully.")

# ------------------------------------------------------------
# Load Autoencoder
# ------------------------------------------------------------
print("\nLoading Autoencoder...")

ae_model = keras.models.load_model(
    AE_MODEL_FILE,
    compile=False
)

print("Autoencoder loaded successfully.")

IF model exists: True
AE model exists: True

Loading Isolation Forest...
Isolation Forest loaded successfully.

Loading Autoencoder...
Autoencoder loaded successfully.


In [16]:
# ============================================================
# ISOLATION FOREST ANOMALY SCORES
# ============================================================

if_val_scores = (
    -if_model.score_samples(X_val)
)

if_test_scores = (
    -if_model.score_samples(X_test)
)

print(
    "IF validation scores:",
    if_val_scores.shape
)

print(
    "IF test scores:",
    if_test_scores.shape
)

print(
    "\nIF validation score range:"
)

print(
    "Min:",
    if_val_scores.min()
)

print(
    "Max:",
    if_val_scores.max()
)

print(
    "Mean:",
    if_val_scores.mean()
)

IF validation scores: (10266,)
IF test scores: (82332,)

IF validation score range:
Min: 0.34999694647262913
Max: 0.6895902093294255
Mean: 0.4010166093607668


In [17]:
# ============================================================
# AUTOENCODER ANOMALY SCORES
# ============================================================

def calculate_ae_scores(
    model,
    X
):

    reconstructed = model.predict(
        X,
        batch_size=1024,
        verbose=0
    )

    reconstruction_error = np.mean(
        np.square(
            X - reconstructed
        ),
        axis=1
    )

    return reconstruction_error


ae_val_scores = calculate_ae_scores(
    ae_model,
    X_val
)

ae_test_scores = calculate_ae_scores(
    ae_model,
    X_test
)

print(
    "AE validation scores:",
    ae_val_scores.shape
)

print(
    "AE test scores:",
    ae_test_scores.shape
)

print(
    "\nAE validation score range:"
)

print(
    "Min:",
    ae_val_scores.min()
)

print(
    "Max:",
    ae_val_scores.max()
)

print(
    "Mean:",
    ae_val_scores.mean()
)

AE validation scores: (10266,)
AE test scores: (82332,)

AE validation score range:
Min: 0.00012109104
Max: 0.2569234
Mean: 0.0054246215


In [18]:
# ============================================================
# SCORE CHECK
# ============================================================

print("Isolation Forest")
print("----------------")

print(
    "Normal test mean:",
    if_test_scores[
        y_test == 0
    ].mean()
)

print(
    "Attack test mean:",
    if_test_scores[
        y_test == 1
    ].mean()
)


print("\nAutoencoder")
print("-----------")

print(
    "Normal test mean:",
    ae_test_scores[
        y_test == 0
    ].mean()
)

print(
    "Attack test mean:",
    ae_test_scores[
        y_test == 1
    ].mean()
)

Isolation Forest
----------------
Normal test mean: 0.4173044853386954
Attack test mean: 0.5688987570766879

Autoencoder
-----------
Normal test mean: 0.009899388
Attack test mean: 0.0806385


In [19]:
# ============================================================
# EMPIRICAL PERCENTILE NORMALISATION
#
# Output:
# approximately 0.0 -> very normal
# approximately 1.0 -> highly unusual compared with validation
# ============================================================

def percentile_normalize(
    reference_scores,
    scores
):

    reference_sorted = np.sort(
        np.asarray(reference_scores)
    )

    ranks = np.searchsorted(
        reference_sorted,
        scores,
        side="right"
    )

    normalized = (
        ranks /
        (len(reference_sorted) + 1)
    )

    return normalized


# ------------------------------------------------------------
# Normalise IF scores
# ------------------------------------------------------------

if_val_norm = percentile_normalize(
    if_val_scores,
    if_val_scores
)

if_test_norm = percentile_normalize(
    if_val_scores,
    if_test_scores
)


# ------------------------------------------------------------
# Normalise AE scores
# ------------------------------------------------------------

ae_val_norm = percentile_normalize(
    ae_val_scores,
    ae_val_scores
)

ae_test_norm = percentile_normalize(
    ae_val_scores,
    ae_test_scores
)


print("Normalized IF validation:")
print(
    "Min:",
    if_val_norm.min(),
    "Max:",
    if_val_norm.max()
)

print("\nNormalized AE validation:")
print(
    "Min:",
    ae_val_norm.min(),
    "Max:",
    ae_val_norm.max()
)

Normalized IF validation:
Min: 9.739943508327652e-05 Max: 0.9999026005649168

Normalized AE validation:
Min: 9.739943508327652e-05 Max: 0.9999026005649168


In [20]:
# ============================================================
# EXAMPLE NORMALIZED SCORES
# ============================================================

score_preview = pd.DataFrame({
    "if_raw_score":
        if_test_scores[:10],

    "if_normalized":
        if_test_norm[:10],

    "ae_raw_score":
        ae_test_scores[:10],

    "ae_normalized":
        ae_test_norm[:10],

    "actual_label":
        y_test[:10]
})

display(
    score_preview
)

,if_raw_score,if_normalized,ae_raw_score,ae_normalized,actual_label
0,0.450713,0.890328,0.001089,0.239797,0
1,0.463685,0.925976,0.002705,0.511250,0
2,0.439728,0.844551,0.001193,0.266095,0
3,0.441019,0.850200,0.000792,0.168988,0
4,0.460208,0.918866,0.002585,0.496932,0
5,0.464486,0.927632,0.001073,0.235122,0
6,0.473945,0.944580,0.002580,0.496348,0
7,0.482098,0.957436,0.003019,0.546119,0
8,0.630057,0.999513,0.131769,0.999318,0
9,0.630057,0.999513,0.131769,0.999318,0


In [21]:
# ============================================================
# HYBRID WEIGHT SETTINGS
# ============================================================

PRIMARY_IF_WEIGHT = 0.50
PRIMARY_AE_WEIGHT = 0.50

weight_candidates = [
    (0.25, 0.75),
    (0.50, 0.50),
    (0.75, 0.25)
]

print(
    "Primary hybrid:"
)

print(
    f"IF weight = {PRIMARY_IF_WEIGHT}"
)

print(
    f"AE weight = {PRIMARY_AE_WEIGHT}"
)

Primary hybrid:
IF weight = 0.5
AE weight = 0.5


In [22]:
# ============================================================
# PRIMARY HYBRID SCORE
# ============================================================

hybrid_val_scores = (
    PRIMARY_IF_WEIGHT
    * if_val_norm
    +
    PRIMARY_AE_WEIGHT
    * ae_val_norm
)

hybrid_test_scores = (
    PRIMARY_IF_WEIGHT
    * if_test_norm
    +
    PRIMARY_AE_WEIGHT
    * ae_test_norm
)


print(
    "Hybrid validation:",
    hybrid_val_scores.shape
)

print(
    "Hybrid test:",
    hybrid_test_scores.shape
)

print(
    "\nHybrid validation range:"
)

print(
    "Min:",
    hybrid_val_scores.min()
)

print(
    "Max:",
    hybrid_val_scores.max()
)

print(
    "Mean:",
    hybrid_val_scores.mean()
)

Hybrid validation: (10266,)
Hybrid test: (82332,)

Hybrid validation range:
Min: 0.003798577968247784
Max: 0.9998052011298335
Mean: 0.500000232445564


In [23]:
# ============================================================
# FALSE-POSITIVE BUDGETS
# ============================================================

false_positive_budgets = {
    "0.5% budget": 0.005,
    "1% budget": 0.01,
    "3% budget": 0.03
}

hybrid_thresholds = {}


for budget_name, budget in (
    false_positive_budgets.items()
):

    threshold = np.quantile(
        hybrid_val_scores,
        1 - budget
    )

    hybrid_thresholds[
        budget_name
    ] = float(threshold)

    val_predictions = (
        hybrid_val_scores
        >= threshold
    ).astype(int)

    validation_alert_rate = (
        val_predictions.mean()
    )

    print(
        f"{budget_name}"
        f" | Threshold: {threshold:.6f}"
        f" | Validation alert rate: "
        f"{validation_alert_rate:.2%}"
    )

0.5% budget | Threshold: 0.990994 | Validation alert rate: 0.51%
1% budget | Threshold: 0.980082 | Validation alert rate: 1.01%
3% budget | Threshold: 0.938938 | Validation alert rate: 3.00%


In [24]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_scores(
    y_true,
    scores,
    threshold
):

    predictions = (
        scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    actual_fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    false_alerts_per_1000 = (
        actual_fpr * 1000
    )

    return {
        "Threshold":
            float(threshold),

        "Precision":
            float(precision),

        "Recall":
            float(recall),

        "F1-score":
            float(f1),

        "Actual FPR":
            float(actual_fpr),

        "False alerts per 1000":
            float(
                false_alerts_per_1000
            ),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp)
    }

In [25]:
# ============================================================
# PRIMARY HYBRID EVALUATION
# ============================================================

hybrid_budget_rows = []


for budget_name, threshold in (
    hybrid_thresholds.items()
):

    result = evaluate_scores(
        y_test,
        hybrid_test_scores,
        threshold
    )

    result[
        "Budget"
    ] = budget_name

    hybrid_budget_rows.append(
        result
    )


hybrid_budget_results = pd.DataFrame(
    hybrid_budget_rows
)


hybrid_budget_results = (
    hybrid_budget_results[
        [
            "Budget",
            "Threshold",
            "Precision",
            "Recall",
            "F1-score",
            "Actual FPR",
            "False alerts per 1000",
            "TN",
            "FP",
            "FN",
            "TP"
        ]
    ]
)


display(
    hybrid_budget_results
)

,Budget,Threshold,Precision,Recall,F1-score,Actual FPR,False alerts per 1000,TN,FP,FN,TP
0,0.5% budget,0.990994,0.954243,0.612768,0.746299,0.036000,36.000000,35668,1332,17554,27778
1,1% budget,0.980082,0.943977,0.625938,0.752742,0.045514,45.513514,35316,1684,16957,28375
2,3% budget,0.938938,0.911084,0.679233,0.778258,0.081216,81.216216,33995,3005,14541,30791


In [26]:
# ============================================================
# RANKING METRICS
# ============================================================

hybrid_roc_auc = roc_auc_score(
    y_test,
    hybrid_test_scores
)

hybrid_pr_auc = average_precision_score(
    y_test,
    hybrid_test_scores
)

print(
    f"Hybrid ROC-AUC: "
    f"{hybrid_roc_auc:.4f}"
)

print(
    f"Hybrid PR-AUC: "
    f"{hybrid_pr_auc:.4f}"
)

Hybrid ROC-AUC: 0.8767
Hybrid PR-AUC: 0.9057


In [27]:
# ============================================================
# DEFAULT OPERATING POINT = 1%
# ============================================================

DEFAULT_BUDGET = "1% budget"

default_hybrid_threshold = (
    hybrid_thresholds[
        DEFAULT_BUDGET
    ]
)

default_hybrid_predictions = (
    hybrid_test_scores
    >= default_hybrid_threshold
).astype(int)


default_hybrid_metrics = evaluate_scores(
    y_test,
    hybrid_test_scores,
    default_hybrid_threshold
)


print(
    "HYBRID IF + AE — DEFAULT 1% BUDGET"
)

print(
    "=================================="
)

print(
    "IF weight:",
    PRIMARY_IF_WEIGHT
)

print(
    "AE weight:",
    PRIMARY_AE_WEIGHT
)

print(
    f"Threshold: "
    f"{default_hybrid_threshold:.6f}"
)

print(
    f"Precision: "
    f"{default_hybrid_metrics['Precision']:.4f}"
)

print(
    f"Recall: "
    f"{default_hybrid_metrics['Recall']:.4f}"
)

print(
    f"F1-score: "
    f"{default_hybrid_metrics['F1-score']:.4f}"
)

print(
    f"Actual FPR: "
    f"{default_hybrid_metrics['Actual FPR']:.4%}"
)

print(
    f"False alerts / 1000: "
    f"{default_hybrid_metrics['False alerts per 1000']:.2f}"
)

print(
    f"ROC-AUC: "
    f"{hybrid_roc_auc:.4f}"
)

print(
    f"PR-AUC: "
    f"{hybrid_pr_auc:.4f}"
)

HYBRID IF + AE — DEFAULT 1% BUDGET
IF weight: 0.5
AE weight: 0.5
Threshold: 0.980082
Precision: 0.9440
Recall: 0.6259
F1-score: 0.7527
Actual FPR: 4.5514%
False alerts / 1000: 45.51
ROC-AUC: 0.8767
PR-AUC: 0.9057


In [28]:
# ============================================================
# INDIVIDUAL MODEL THRESHOLDS
# ============================================================

if_thresholds = {}
ae_thresholds = {}


for budget_name, budget in (
    false_positive_budgets.items()
):

    if_thresholds[
        budget_name
    ] = float(
        np.quantile(
            if_val_scores,
            1 - budget
        )
    )

    ae_thresholds[
        budget_name
    ] = float(
        np.quantile(
            ae_val_scores,
            1 - budget
        )
    )


print(
    "IF thresholds:",
    if_thresholds
)

print()

print(
    "AE thresholds:",
    ae_thresholds
)

IF thresholds: {'0.5% budget': 0.5676495685072951, '1% budget': 0.5460537691550149, '3% budget': 0.4966390963721801}

AE thresholds: {'0.5% budget': 0.04577397182583809, '1% budget': 0.036467839032411575, '3% budget': 0.023754458874464035}


In [29]:
# ============================================================
# MODEL COMPARISON — DEFAULT 1% BUDGET
# ============================================================

if_metrics_1pct = evaluate_scores(
    y_test,
    if_test_scores,
    if_thresholds[
        "1% budget"
    ]
)

ae_metrics_1pct = evaluate_scores(
    y_test,
    ae_test_scores,
    ae_thresholds[
        "1% budget"
    ]
)

hybrid_metrics_1pct = evaluate_scores(
    y_test,
    hybrid_test_scores,
    hybrid_thresholds[
        "1% budget"
    ]
)


if_roc_auc = roc_auc_score(
    y_test,
    if_test_scores
)

if_pr_auc = average_precision_score(
    y_test,
    if_test_scores
)


ae_roc_auc = roc_auc_score(
    y_test,
    ae_test_scores
)

ae_pr_auc = average_precision_score(
    y_test,
    ae_test_scores
)


model_comparison = pd.DataFrame([
    {
        "Model":
            "Isolation Forest",

        "Precision":
            if_metrics_1pct[
                "Precision"
            ],

        "Recall":
            if_metrics_1pct[
                "Recall"
            ],

        "F1-score":
            if_metrics_1pct[
                "F1-score"
            ],

        "Actual FPR":
            if_metrics_1pct[
                "Actual FPR"
            ],

        "False alerts per 1000":
            if_metrics_1pct[
                "False alerts per 1000"
            ],

        "ROC-AUC":
            if_roc_auc,

        "PR-AUC":
            if_pr_auc
    },

    {
        "Model":
            "Dense Autoencoder",

        "Precision":
            ae_metrics_1pct[
                "Precision"
            ],

        "Recall":
            ae_metrics_1pct[
                "Recall"
            ],

        "F1-score":
            ae_metrics_1pct[
                "F1-score"
            ],

        "Actual FPR":
            ae_metrics_1pct[
                "Actual FPR"
            ],

        "False alerts per 1000":
            ae_metrics_1pct[
                "False alerts per 1000"
            ],

        "ROC-AUC":
            ae_roc_auc,

        "PR-AUC":
            ae_pr_auc
    },

    {
        "Model":
            "Hybrid IF + AE",

        "Precision":
            hybrid_metrics_1pct[
                "Precision"
            ],

        "Recall":
            hybrid_metrics_1pct[
                "Recall"
            ],

        "F1-score":
            hybrid_metrics_1pct[
                "F1-score"
            ],

        "Actual FPR":
            hybrid_metrics_1pct[
                "Actual FPR"
            ],

        "False alerts per 1000":
            hybrid_metrics_1pct[
                "False alerts per 1000"
            ],

        "ROC-AUC":
            hybrid_roc_auc,

        "PR-AUC":
            hybrid_pr_auc
    }
])


display(
    model_comparison
)

,Model,Precision,Recall,F1-score,Actual FPR,False alerts per 1000,ROC-AUC,PR-AUC
0,Isolation Forest,0.940079,0.617070,0.745072,0.048189,48.189189,0.851001,0.897443
1,Dense Autoencoder,0.936372,0.632710,0.755157,0.052676,52.675676,0.878460,0.901098
2,Hybrid IF + AE,0.943977,0.625938,0.752742,0.045514,45.513514,0.876671,0.905663


In [30]:
# ============================================================
# HYBRID OPERATING-POINT PREDICTIONS
# ============================================================

hybrid_threshold_005 = (
    hybrid_thresholds[
        "0.5% budget"
    ]
)

hybrid_threshold_01 = (
    hybrid_thresholds[
        "1% budget"
    ]
)

hybrid_threshold_03 = (
    hybrid_thresholds[
        "3% budget"
    ]
)


hybrid_pred_005 = (
    hybrid_test_scores
    >= hybrid_threshold_005
).astype(int)


hybrid_pred_01 = (
    hybrid_test_scores
    >= hybrid_threshold_01
).astype(int)


hybrid_pred_03 = (
    hybrid_test_scores
    >= hybrid_threshold_03
).astype(int)

In [33]:
# ============================================================
# DASHBOARD-READY ROW LEVEL RESULTS
# ============================================================

# Individual model predictions at default 1% budget
if_pred_01 = (
    if_test_scores >= if_thresholds["1% budget"]
).astype(int)

ae_pred_01 = (
    ae_test_scores >= ae_thresholds["1% budget"]
).astype(int)


# ------------------------------------------------------------
# IF + AE agreement
# ------------------------------------------------------------

def agreement_label(if_pred, ae_pred):

    if if_pred == 1 and ae_pred == 1:
        return "Both anomaly"

    elif if_pred == 1 and ae_pred == 0:
        return "IF only"

    elif if_pred == 0 and ae_pred == 1:
        return "AE only"

    else:
        return "Neither"


agreement = [
    agreement_label(i, a)
    for i, a in zip(
        if_pred_01,
        ae_pred_01
    )
]


# ------------------------------------------------------------
# Create dashboard dataframe
# ------------------------------------------------------------

dashboard_results = pd.DataFrame({

    "record_id":
        np.arange(len(y_test)),

    "actual_label":
        y_test,

    "actual_class":
        np.where(
            y_test == 1,
            "Attack",
            "Normal"
        ),

    # Isolation Forest
    "if_score":
        if_test_scores,

    "if_score_normalized":
        if_test_norm,

    "if_prediction_1pct":
        if_pred_01,

    "if_status_1pct":
        np.where(
            if_pred_01 == 1,
            "Anomaly",
            "Normal"
        ),

    # Autoencoder
    "ae_score":
        ae_test_scores,

    "ae_score_normalized":
        ae_test_norm,

    "ae_prediction_1pct":
        ae_pred_01,

    "ae_status_1pct":
        np.where(
            ae_pred_01 == 1,
            "Anomaly",
            "Normal"
        ),

    # Agreement
    "if_ae_agreement":
        agreement,

    # Hybrid
    "hybrid_score":
        hybrid_test_scores,

    "hybrid_prediction_0.5pct":
        hybrid_pred_005,

    "hybrid_status_0.5pct":
        np.where(
            hybrid_pred_005 == 1,
            "Anomaly",
            "Normal"
        ),

    "hybrid_prediction_1pct":
        hybrid_pred_01,

    "hybrid_status_1pct":
        np.where(
            hybrid_pred_01 == 1,
            "Anomaly",
            "Normal"
        ),

    "hybrid_prediction_3pct":
        hybrid_pred_03,

    "hybrid_status_3pct":
        np.where(
            hybrid_pred_03 == 1,
            "Anomaly",
            "Normal"
        ),

    # Default operating point = 1%
    "default_threshold":
        default_hybrid_threshold,

    "default_prediction":
        hybrid_pred_01,

    "default_status":
        np.where(
            hybrid_pred_01 == 1,
            "Anomaly",
            "Normal"
        ),

    "correct_prediction":
        (
            hybrid_pred_01 == y_test
        ).astype(int)
})


print(
    "Dashboard rows:",
    len(dashboard_results)
)

print(
    "Dashboard columns:",
    len(dashboard_results.columns)
)

display(
    dashboard_results.head()
)

Dashboard rows: 82332
Dashboard columns: 23


,record_id,actual_label,actual_class,if_score,if_score_normalized,if_prediction_1pct,if_status_1pct,ae_score,ae_score_normalized,ae_prediction_1pct,...,hybrid_prediction_0.5pct,hybrid_status_0.5pct,hybrid_prediction_1pct,hybrid_status_1pct,hybrid_prediction_3pct,hybrid_status_3pct,default_threshold,default_prediction,default_status,correct_prediction
0,0,0,Normal,0.450713,0.890328,0,Normal,0.001089,0.239797,0,...,0,Normal,0,Normal,0,Normal,0.980082,0,Normal,1
1,1,0,Normal,0.463685,0.925976,0,Normal,0.002705,0.511250,0,...,0,Normal,0,Normal,0,Normal,0.980082,0,Normal,1
2,2,0,Normal,0.439728,0.844551,0,Normal,0.001193,0.266095,0,...,0,Normal,0,Normal,0,Normal,0.980082,0,Normal,1
3,3,0,Normal,0.441019,0.850200,0,Normal,0.000792,0.168988,0,...,0,Normal,0,Normal,0,Normal,0.980082,0,Normal,1
4,4,0,Normal,0.460208,0.918866,0,Normal,0.002585,0.496932,0,...,0,Normal,0,Normal,0,Normal,0.980082,0,Normal,1


In [34]:
# ============================================================
# SAVE DASHBOARD + SUMMARY FILES
# ============================================================

from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "intentmap-nids/intentmap-nids/"
    "results/hybrid"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Main dashboard file
dashboard_file = (
    OUTPUT_DIR /
    "hybrid_results_for_dashboard.csv"
)

dashboard_results.to_csv(
    dashboard_file,
    index=False
)


# Model comparison summary
comparison_file = (
    OUTPUT_DIR /
    "if_ae_hybrid_comparison.csv"
)

model_comparison.to_csv(
    comparison_file,
    index=False
)


# Hybrid threshold-budget results
budget_file = (
    OUTPUT_DIR /
    "hybrid_budget_results.csv"
)

hybrid_budget_results.to_csv(
    budget_file,
    index=False
)


print("FILES SAVED")
print("===========")

print(dashboard_file)
print(comparison_file)
print(budget_file)

FILES SAVED
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/hybrid/hybrid_results_for_dashboard.csv
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/hybrid/if_ae_hybrid_comparison.csv
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/hybrid/hybrid_budget_results.csv


In [35]:
# ============================================================
# SAVE DASHBOARD + SUMMARY FILES
# ============================================================

from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "intentmap-nids/intentmap-nids/"
    "results/hybrid"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Main dashboard file
dashboard_file = (
    OUTPUT_DIR /
    "hybrid_results_for_dashboard.csv"
)

dashboard_results.to_csv(
    dashboard_file,
    index=False
)


# Model comparison summary
comparison_file = (
    OUTPUT_DIR /
    "if_ae_hybrid_comparison.csv"
)

model_comparison.to_csv(
    comparison_file,
    index=False
)


# Hybrid threshold-budget results
budget_file = (
    OUTPUT_DIR /
    "hybrid_budget_results.csv"
)

hybrid_budget_results.to_csv(
    budget_file,
    index=False
)


print("FILES SAVED")
print("===========")

print(dashboard_file)
print(comparison_file)
print(budget_file)

FILES SAVED
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/hybrid/hybrid_results_for_dashboard.csv
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/hybrid/if_ae_hybrid_comparison.csv
/content/drive/MyDrive/intentmap-nids/intentmap-nids/results/hybrid/hybrid_budget_results.csv
